# Trabalho Prático 02: S-AES e AES/Modos de Operação

## Objetivo
Desenvolver uma implementação do Simplified AES (S-AES), utilizando blocos de 16 bits e composto por 2 rodadas, seguindo os princípios fundamentais do AES original.

## Introdução
Falar sobre o AES

Comparar com o DES

Falar sobre o S-AES

Comparativo entre o S-AES e o AES oficial


### A classe `Bin`

A classe `Bin` foi projetada para representar e manipular números binários de tamanho fixo em Python. Ela encapsula um valor inteiro (`value`) e seu tamanho em bits (`size`), oferecendo diversas operações e conversões úteis para aplicações envolvendo criptografia, álgebra booleana, manipulação de bits e codificação.

**Principais componentes**

- **Inicialização**
    - `__init__(self, value, size)`: Inicializa um objeto `Bin` com o valor inteiro e o tamanho especificado em bits.

- **Representações e formatação**
    - `__repr__` e `__str__`: Retornam representações em hexadecimal para debug e exibição.
    - `__format__`: Permite formatar o objeto em diferentes bases (binária, hexadecimal, base64) com suporte a agrupamento de dígitos.

- **Operações bit a bit**
    - `__xor__`, `__or__`, `__and__`: Realizam operações lógicas bit a bit entre dois objetos `Bin`, garantindo que ambos tenham o mesmo tamanho.
    - `__lshift__`, `__rshift__`: Realizam deslocamentos circulares dos bits para a esquerda ou direita.
    - `xor(self, *bins)`: Aplica XOR sucessivo entre o objeto e múltiplos `Bin`.

- **Operações aritméticas**
    - `__add__`, `__mul__`: Permitem somar e multiplicar dois objetos `Bin`, ajustando o tamanho do resultado de acordo com o número de bits necessários.
    - `finite_field_mul`: Realiza multiplicação no campo finito GF(2^4), com redução modular pelo polinômio 0x13 (útil em criptografia, como no AES).

- **Manipulação e transformação**
    - `copy`: Cria uma cópia do objeto.
    - `to_list`: Converte o valor em uma lista de bits.
    - `to_bytes`: Converte o valor para um array de bytes.
    - `to_str`: Converte o valor para uma string em formato binário, hexadecimal ou base64, com agrupamento opcional.
    - `extract`: Extrai bits em posições específicas, retornando um novo `Bin`.
    - `halve`: Divide o `Bin` em duas metades.
    - `split`: Divide o valor em pedaços menores de tamanho fixo.
    - `swap`: Troca as metades esquerda e direita do valor.
    - `extend`: Anexa os bits de outro `Bin` ao final do valor atual.
    - `fuse`: Junta vários objetos `Bin` em um único.
    - `from_hex`: Cria um `Bin` a partir de uma string hexadecimal.

- **Validação**
    - `assert_same_size`: Verifica se dois objetos `Bin` têm o mesmo tamanho antes de operações bit a bit.


In [49]:
import base64
from typing import List


# Representa um número binário com tamanho fixo
class Bin:
    def __init__(self, value: int, size: int):
        self.value = value  # Valor inteiro
        self.size = size  # Tamanho em bits
        
    # Representação para debug (em hexadecimal)
    def __repr__(self):
        return f"Bin(value={self}, size={self.size})"

    # Representação em string (em hexadecimal)
    def __str__(self, type="x", group_size=0):
        hex_digits = (self.size + 3) // 4
        return f"0x{self.value:0{hex_digits}X}"

    # Format_spec example: "x", "b", or "b_4" (for grouping)
    def __format__(self, format_spec):
        if not format_spec:
            return str(self)
        parts = format_spec.split("_")
        fmt = parts[0]
        group_size = int(parts[1]) if len(parts) > 1 else 0

        if fmt in ("b", "B", "x", "X", "base64"):
            return self.to_str(fmt, group_size)
        else:
            return str(self)

    # Verifica igualdade com outro objeto Bin
    def __eq__(self, other):
        if isinstance(other, Bin):
            return self.value == other.value and self.size == other.size
        return False

    # Verifica desigualdade com outro objeto Bin
    def __ne__(self, other):
        if isinstance(other, Bin):
            return not self.__eq__(other)
        return True

    # Faz o XOR bit a bit entre dois objetos Bin
    def __xor__(self, other: "Bin") -> "Bin":
        self.assert_same_size(other)
        return Bin(self.value ^ other.value, self.size)

    # Faz o OR bit a bit entre dois objetos Bin
    def __or__(self, other: "Bin") -> "Bin":
        self.assert_same_size(other)
        return Bin(self.value | other.value, self.size)

    # Faz o AND bit a bit entre dois objetos Bin
    def __and__(self, other: "Bin") -> "Bin":
        self.assert_same_size(other)
        return Bin(self.value & other.value, self.size)

    # Faz o MUL entre dois objetos Bin
    def __mul__(self, other: "Bin") -> "Bin":
        result_value = self.value * other.value
        result_size = result_value.bit_length() or 1
        return Bin(result_value, result_size)

    # Faz o ADD entre dois objetos Bin
    def __add__(self, other: "Bin") -> "Bin":
        result_value = self.value + other.value
        result_size = result_value.bit_length() or 1
        return Bin(result_value, result_size)

    # Faz o circular shift à esquerda 
    def __lshift__(self, shift_value: int) -> "Bin":
        shift_value %= self.size
        discarded = self.value >> (self.size - shift_value)
        mask = (1 << self.size) - 1
        shifted = (self.value << shift_value) & mask
        result = shifted | discarded
        return Bin(result, self.size)

    # Faz o circular shift à direita
    def __rshift__(self, shift_value: int) -> "Bin":
        shift_value %= self.size
        discarded = (self.value << (self.size - shift_value)) & ((1 << self.size) - 1)
        shifted = self.value >> shift_value
        result = shifted | discarded
        return Bin(result, self.size)

    # Faz a cópia do objeto Bin
    def copy(self):
        return Bin(self.value, self.size)

    # Converte o valor Bin para uma lista de bits
    def to_list(self) -> List[int]:
        return [(self.value >> i) & 1 for i in reversed(range(self.size))]

    def to_bytes(self):
        byte_length = (self.size + 7) // 8
        return self.value.to_bytes(byte_length, byteorder='big')

    # Converte o valor Bin para uma string formatada
    def to_str(self, fmt="x", group_size=0):
        if fmt in ("x", "X"):
            hex_digits = (self.size + 3) // 4
            raw_hex = f"{self.value:0{hex_digits}{fmt}}"
            if group_size <= 0:
                return f"0x{raw_hex}"
            else:
                # Group hex digits from right to left (common style)
                groups = []
                for i in range(len(raw_hex), 0, -group_size):
                    start = max(0, i - group_size)
                    groups.append(raw_hex[start:i])
                grouped = "_".join(reversed(groups))
                return f"0x{grouped}"

        elif fmt in ("b", "B"):
            raw_bin = f"{self.value:0{self.size}b}"
            if group_size <= 0:
                return f"0b{raw_bin}"
            else:
                grouped = "_".join(
                    raw_bin[i : i + group_size]
                    for i in range(0, len(raw_bin), group_size)
                )
                return f"0b{grouped}"

        elif fmt == "base64":
            raw_bytes = self.to_bytes()
            b64_str = base64.b64encode(raw_bytes).decode("ascii")
            if group_size <= 0:
                return b64_str
            else:
                grouped = " ".join(
                    b64_str[i : i + group_size]
                    for i in range(0, len(b64_str), group_size)
                )
                return grouped
        else:
            raise ValueError(f"Unsupported format: {fmt}")

    # Realiza operação XOR bit a bit entre vários Bin
    def xor(self, *bins: "Bin"):
        for bin in bins:
            self.assert_same_size(bin)
            self.value ^= bin.value
        return self

    # Extrai bits com base em uma tabela de posições e retorna como novo Bin
    def extract(self, table: List[List[int]]):
        extracted_bits = []
        for row in table:
            for bit_pos in row:
                bit = (self.value >> (self.size - bit_pos)) & 1
                extracted_bits.append(bit)
        result = 0
        for bit in extracted_bits:
            result = (result << 1) | bit
        return Bin(result, len(extracted_bits))

    # Divide o valor Bin em duas metades e retorna como dois objetos Bin
    def halve(self):
        half = (self.size + 1) // 2
        left = (self.value >> half) & ((1 << half) - 1)
        right = self.value & ((1 << half) - 1)
        return Bin(left, half), Bin(right, half)

    # Divide o Bin em pedaços menores de tamanho fixo
    def split(self, chunk_size: int):
        remainder = self.size % chunk_size
        if remainder != 0:
            # Calculate how many bits to pad
            pad_size = chunk_size - remainder
            self.value <<= pad_size  # Shift left to pad with zeros
            self.size += pad_size

        chunks: List[Bin] = []
        for i in range(0, self.size, chunk_size):
            chunk_value = (self.value >> (self.size - i - chunk_size)) & (
                (1 << chunk_size) - 1
            )
            chunks.append(Bin(chunk_value, chunk_size))
        return chunks

    # Troca as metades esquerda e direita de um Bin
    def swap(self):
        left, right = self.halve()
        right.extend(left)
        self.value = right.value

    # Anexa os bits de outro Bin ao final deste Bin
    def extend(self, bin: "Bin"):
        self.value = (self.value << bin.size) | bin.value
        self.size += bin.size

    def assert_same_size(self, bin: "Bin"):
        if self.size != bin.size:
            raise ValueError("Bin sizes must match for bitwise operations")

    # Junta vários objetos Bin em um só
    @staticmethod
    def fuse(*bins: "Bin"):
        total_value = 0
        total_size = 0
        for b in bins:
            total_value = (total_value << b.size) | b.value
            total_size += b.size
        return Bin(total_value, total_size)

    # Converte uma str em formato '0x' para um Bin
    @staticmethod
    def from_hex(hex_str: str):
        return Bin(int(hex_str, 16), len(hex_str) * 4)
    
    # Multiplica dois objetos Bin em um campo finito (GF(2^4) com módulo 0x13)
    @staticmethod
    def finite_field_mul(a: "Bin", b: "Bin") -> "Bin":
        assert a.size == b.size == 4, "Operands must be 4-bit for GF(2^4)"
        
        modulus = 0x13
        result = 0
        a_val = a.value
        b_val = b.value
        for _ in range(4):  # GF(2^4)
            if b_val & 1:
                result ^= a_val
            b_val >>= 1
            a_val <<= 1
            if a_val & 0x10:      # Check if x^4 is set (overflow)
                a_val ^= modulus  # Modular reduction

        return Bin(result & 0xF, 4)  # Ensure result is 4 bits

### A classe `BinBlock`


A classe `BinBlock` representa uma estrutura bidimensional (um bloco 2x2) de nibbles (grupos de 4 bits), construída a partir de um número binário de 16 bits encapsulado em um objeto `Bin`. Essa estrutura é útil em operações que trabalham com blocos de dados, como cifradores em blocos ou manipulação de dados em matrizes binárias.

**Principais componentes**

- **Inicialização**
    - `__init__(self, bin=Bin(0x0, 16))`: O construtor recebe um objeto `Bin` com exatamente 16 bits. Esse valor é dividido em quatro nibbles (4 bits cada), organizados em um bloco 2x2 da seguinte forma:

      ```
      [[nibble0, nibble2],
       [nibble1, nibble3]]
      ```

    Isso permite acesso e manipulação de cada pedaço de 4 bits individualmente no contexto do bloco.

- **Representações**
    - `__repr__`: Fornece uma representação detalhada e estruturada para debug, mostrando o bloco como uma lista de listas.
    - `__str__`: Retorna uma string formatada com os nibbles em hexadecimal, organizados em linhas.

- **Operadores bit a bit**
    - `__xor__`, `__or__`, `__and__`: Realizam operações bit a bit entre dois blocos (`BinBlock`), aplicando a operação correspondente elemento a elemento. O resultado é um novo `BinBlock` com os valores calculados.

- **Manipulação de linhas e colunas**
    - `shift_line_left(l, amt)`: Faz um deslocamento circular à esquerda em uma linha do bloco (linha `l`) por uma quantidade `amt`. A linha é primeiro fundida em um único `Bin`, deslocada e depois dividida novamente em nibbles.
    - `get_column(c)`: Retorna a coluna `c` como uma lista de dois `Bin`. Obs.: O índice da coluna é limitado ao intervalo válido.
    - `get_columns()`: Retorna todas as colunas do bloco como uma lista de listas, permitindo iteração coluna por coluna.

- **Outros métodos**
    - `copy()`: Cria uma cópia do bloco, preservando a estrutura e os valores.
    - `to_bin()`: Concatena os nibbles do bloco em um único `Bin` de 16 bits. A ordem dos nibbles na fusão respeita o padrão:

      ```
      [nibble0, nibble2, nibble1, nibble3]
      ```

    garantindo consistência com a forma como o bloco foi construído.

In [50]:
class BinBlock:
    block: List[List[Bin]] = []

    def __init__(self, bin=Bin(0x0, 16)):
        if bin.size != 16:
            raise ValueError("BinBlock requires a 16-bit Bin") 

        nibbles = bin.split(4)
        block = [
            [nibbles[0], nibbles[2]],
            [nibbles[1], nibbles[3]]
        ]


        self.block = block
        self.nibbles = 4

    # Representação para debug
    def __repr__(self):
        rows = ",\n  ".join([str(row) for row in self.block])
        return f"BinBlock([\n  {rows}\n])"

    # Representação em string
    def __str__(self):
        rows = "\n".join([" ".join([str(b) for b in row]) for row in self.block])
        return rows

    # Verifica igualdade com outro BinBlock
    def __eq__(self, other):
        if not isinstance(other, BinBlock):
            return False
        return self.block == other.block and self.nibbles == other.nibbles

    # Faz o XOR bit a bit entre dois BinBlocks
    def __xor__(self, other: "BinBlock"):
        if not isinstance(other, BinBlock):
            raise TypeError("Operand must be a BinBlock")

        result_block = []
        for row_self, row_other in zip(self.block, other.block):
            result_row = [a ^ b for a, b in zip(row_self, row_other)]
            result_block.append(result_row)

        result = BinBlock.__new__(BinBlock)
        result.block = result_block
        result.nibbles = self.nibbles
        return result

    # Faz o OR bit a bit entre dois BinBlocks
    def __or__(self, other: "BinBlock"):
        if not isinstance(other, BinBlock):
            raise TypeError("Operand must be a BinBlock")

        result_block = []
        for row_self, row_other in zip(self.block, other.block):
            result_row = [a | b for a, b in zip(row_self, row_other)]
            result_block.append(result_row)

        result = BinBlock.__new__(BinBlock)
        result.block = result_block
        result.nibbles = self.nibbles
        return result

    # Faz o AND bit a bit entre dois BinBlocks
    def __and__(self, other: "BinBlock"):
        if not isinstance(other, BinBlock):
            raise TypeError("Operand must be a BinBlock")

        result_block = []
        for row_self, row_other in zip(self.block, other.block):
            result_row = [a & b for a, b in zip(row_self, row_other)]
            result_block.append(result_row)

        result = BinBlock.__new__(BinBlock)
        result.block = result_block
        result.nibbles = self.nibbles
        return result

    # Faz o shift circular à esquerda de uma linha específica
    def shift_line_left(self, l:int, amt:int):
        line = Bin.fuse(*self.block[l])
        line = line << amt
        self.block[l] = line.split(4)

    # Retorna uma coluna específica do bloco como uma lista de Bin
    def get_column(self, c: int) -> List[Bin]:
        if c not in range(4):
            raise IndexError("Column index must be between 0 and 3")
        return [self.block[row][c] for row in range(2)]

    # Retorna todas as colunas do bloco como uma lista de listas de Bin
    def get_columns(self) -> List[List[Bin]]:
        return [[self.block[row][c] for row in range(2)] for c in range(2)]

    # Retorna uma cópia do BinBlock
    def copy(self) -> "BinBlock":
        new_block = BinBlock()
        new_block.block = self.block.copy()
        new_block.nibbles = self.nibbles
        return new_block

    # Converte o BinBlock para um único Bin
    def to_bin(self) -> Bin:
        flat = [b for row in self.block for b in row]
        ordered = [flat[0], flat[2], flat[1], flat[3]]
        return Bin.fuse(*ordered)

A classe Logger implementa um sistema simples de registro de logs organizados por título ou fase, facilitando a depuração e o rastreamento de eventos ou estados intermediários em um programa. O corpo principal da classe consiste em um dicionário (`self.logs`) que armazena listas de tuplas, onde cada tupla contém uma descrição e o objeto registrado.

**Principais métodos:**

- `log(description: str, obj, title: str = "default")`:  
    Adiciona uma entrada de log sob um título específico. Cada entrada é composta por uma descrição e o objeto a ser registrado (que pode ser qualquer tipo, como variáveis, estruturas de dados ou resultados intermediários).

- `show(title: str = None)`:  
    Exibe os logs registrados. Se um título for fornecido, mostra apenas os logs daquela fase; caso contrário, exibe todos os logs agrupados por título. Cada log é apresentado com seu índice, descrição e representação do objeto.

Essa abordagem modular permite acompanhar facilmente o fluxo de execução e analisar o estado interno do sistema em diferentes etapas, facilitando a depuração e validação de cada fase.

In [51]:
class Logger:
    def __init__(self):
        self.logs = {}

    def log(self, description: str, obj, title: str = "default"):
        if title not in self.logs:
            self.logs[title] = []
        self.logs[title].append((description, obj))

    def show(self, title: str = None):
        if title:
            logs = self.logs.get(title, [])
            print(f"=== Logs for phase '{title}' ===\n")
            for i, (desc, obj) in enumerate(logs, 1):
                print(f"[{i}] {desc}:\n{str(obj)}\n")
        else:
            for title, entries in self.logs.items():
                print(f"=== Logs for phase '{title}' ===\n")
                for i, (desc, obj) in enumerate(entries, 1):
                    print(f"[{i}] {desc}:\n{str(obj)}\n")

## Desenvolvimento

### A classe `S_AES`
===============

A classe `S_AES` representa uma implementação didática e simplificada do algoritmo de criptografia AES (Advanced Encryption Standard), conhecida como S-AES (Simplified AES). Seu objetivo é permitir o estudo e a aplicação dos conceitos fundamentais do AES em blocos de tamanho reduzido (por exemplo, 16 bits), mantendo as principais operações do algoritmo real. Essa versão é útil para fins educacionais e para visualização do funcionamento interno do AES.

### Principais características

- **Inicialização**
    - Recebe um tamanho de chave (padrão: 16 bits) e um modo de operação.
    - Valida o tamanho da chave, define o número de rodadas e inicializa um objeto `Logger` para registrar o estado das operações.

- **Método principal**
    - `encrypt(plaintext: str, key: Bin, ...)`  
      Recebe o texto plano como string e a chave como `Bin`. Converte o texto para binário (`convert_to_bin`) e chama a função de criptografia no modo selecionado (atualmente apenas ECB).

- **Conversão de texto**
    - `convert_to_bin(plaintext: str) -> Bin`  
      Transforma uma string em uma representação binária (`Bin`), codificando em UTF-8 e convertendo os bytes em inteiro.

- **Criptografia de bloco**
    - `aes_ecb_encrypt(plaintext: Bin, key: Bin) -> Bin`  
      Divide o texto em blocos de 16 bits e aplica a criptografia bloco a bloco no modo ECB.

    - `encrypt_block(plaintext: Bin, key: Bin) -> Bin`  
      Criptografa um único bloco: realiza o AddRoundKey inicial, depois executa as rodadas principais usando o método `transform`.

- **Expansão de chave**
    - `key_expansion(key: Bin)`  
      Gera as subchaves para cada rodada. Utiliza a função auxiliar `g`, que rotaciona nibbles, aplica a S-box e combina com a constante de rodada (`r_con`) via XOR. As subchaves são armazenadas como blocos e registradas no logger.

- **Transformações de rodada**
    - `transform(plt: BinBlock, subkey: BinBlock, round: int)`  
      Executa as operações de uma rodada:  
      1. SubBytes (`sub_nibbles`)  
      2. ShiftRows (`shift_rows`)  
      3. MixColumns (`mix_columns`, exceto na última rodada)  
      4. AddRoundKey  
      Cada etapa é registrada no logger.

- **Operações internas**
    - `sub_nibbles(plt: BinBlock) -> BinBlock`  
      Aplica a S-box a cada nibble do bloco.
    - `shift_rows(plt: BinBlock) -> BinBlock`  
      Realiza deslocamento circular na segunda linha do bloco.
    - `mix_columns(plt: BinBlock) -> BinBlock`  
      Mistura as colunas via multiplicação no campo finito.
    - `sbox(nibble: Bin) -> Bin`  
      Implementa a substituição não linear (SubBytes) usando a S-box.

- **Validação**
    - `check_key_size(key: Bin)`  
      Garante que a chave tem o tamanho esperado, lançando exceção caso contrário.

---

Em resumo, a classe `S_AES` implementa uma versão compacta do AES, contendo as principais operações do algoritmo e fornecendo uma infraestrutura para registro e análise das etapas do processo de criptografia, sendo adequada para estudos e simulações.

In [52]:
r_con = [
    0x80, 0x30
]

mix_columns_table = [
    [1, 4],
    [4, 1],
]

substitution_box = [
    [0x9, 0x4, 0xA, 0xB,],
    [0xD, 0x1, 0x8, 0x5,],
    [0x6, 0x2, 0x0, 0x3,],
    [0xC, 0xE, 0xF, 0x7,],
]

sbox_inv = [
    [0xA, 0x5, 0x9, 0xB,],
    [0x1, 0x7, 0x8, 0xF,],
    [0x6, 0x0, 0x2, 0x3,],
    [0xC, 0x4, 0xD, 0xE,],
]

In [ ]:
class S_AES:

    def __init__(self, key_size=16, mode="CBC"):
        if key_size not in [16, 128, 192, 256]:
            raise ValueError("Invalid key size for AES")

        round_numbers = {16: 2, 128: 10, 192: 12, 256: 14}

        self.logger = Logger()
        self.key_size = key_size
        self.rounds = round_numbers[key_size]
        self.mode = mode

    @staticmethod
    def sbox(nibble: Bin):
        if nibble.size != 4:
            raise ValueError("Argument must be a nibble (4 bits)")

        l, c = [b.value for b in nibble.halve()]
        return Bin(substitution_box[l][c], 4)

    def key_expansion(self, key: Bin):
        def g(w: Bin, r: int):
            rotw = w << 4
            nibbles = rotw.split(4)
            nibbles = list(map(S_AES.sbox, nibbles))
            subw = Bin.fuse(*nibbles)
            rcon = Bin(r_con[r], 8)
            g_result = subw ^ rcon
            return g_result

        key_block = BinBlock(key)
        columns = key_block.get_columns()
        w0 = Bin.fuse(*columns[0])
        w1 = Bin.fuse(*columns[1])
        words: List[List[Bin]] = [[w0,w1]]

        round_keys = [BinBlock(Bin.fuse(*words[0]))]
        self.logger.log(f"Round Key 0", round_keys[0], title="Key Expansion")
        for i in range(2):
            #print(words)
            w0, w1 = words[i]
            w2 = w0 ^ g(w1, i)
            w3 = w2 ^ w1


            words.append([w2, w3])
            subkey = BinBlock(Bin.fuse(w2, w3))
            round_keys.append(subkey)
            self.logger.log(f"Round Key {i + 1}", subkey, title="Key Expansion")
        return round_keys

    @staticmethod
    def sub_nibbles(plt: BinBlock) -> BinBlock:
        block = plt.block
        cipher_block = BinBlock()
        for i in range(2):
            for j in range(2):
                s = S_AES.sbox(block[i][j])
                cipher_block.block[i][j] = s

        return cipher_block

    @staticmethod
    def shift_rows(plt: BinBlock):
        cipher_block = plt.copy()
        cipher_block.shift_line_left(1, 4)
        return cipher_block

    @staticmethod
    def mix_columns(plt: BinBlock) -> BinBlock:
        r = Bin(0, 0)
        mct = [[Bin(k, 4) for k in l] for l in mix_columns_table]

        for c in plt.get_columns():
            for l in mct:
                nibble = []
                for i in range(2):
                    nibble.append(Bin.finite_field_mul(l[i], c[i]))
                r.extend(Bin.xor(*nibble))

        return BinBlock(r)

    def transform(self, plt: BinBlock, subkey: BinBlock, round: int):
        last_round = round == self.rounds
        title = f"round {round}"
        self.logger.log("Round key", subkey, title=title)
        self.logger.log("Before Transformations", plt, title=title)

        block = S_AES.sub_nibbles(plt)
        self.logger.log("After SubBytes", block, title=title)

        block = S_AES.shift_rows(block)
        self.logger.log("After ShiftRows", block, title=title)

        if not last_round:
            block = S_AES.mix_columns(block)
            self.logger.log("After MixColumns", block, title=title)

        block = block ^ subkey
        self.logger.log("After AddRoundKey", block, title=title)
        return block

    def encrypt_block(self, plaintext: Bin, key: Bin) -> Bin:
        plt_block = BinBlock(plaintext)
        rnd_keys = self.key_expansion(key)
        

        # Round 0
        cipher_block = plt_block ^ rnd_keys[0]

        self.logger.log("Plaintext block", plt_block, title="round 0")
        self.logger.log("Initial round key", rnd_keys[0], title="round 0")
        self.logger.log("After AddRoundKey", cipher_block, title="round 0")

        # Main rounds: 1 a 10
        for i in range(1, self.rounds + 1):
            cipher_block = self.transform(cipher_block, rnd_keys[i], i)

        return cipher_block.to_bin()

    def decrypt_block(self, ciphertext: Bin, key: Bin) -> Bin:
        pass

    def aes_ecb_encrypt(self, plaintext: Bin, key: Bin) -> Bin:
        chunks = plaintext.split(16)
        blocks: List[Bin] = []

        for chunk in chunks: 
            blocks.append(self.encrypt_block(chunk, key))

        ciphertext = Bin.fuse(*blocks)
        return ciphertext


    def convert_to_bin(self, plaintext: str) -> Bin:
        plaintext_bytes = plaintext.encode('utf-8')
        plaintext_int = int.from_bytes(plaintext_bytes, byteorder='big')
        plaintext_size = len(plaintext_bytes) * 8
        plaintext_bin = Bin(plaintext_int, plaintext_size)
        return plaintext_bin
    
    def encrypt(self, plaintext: str, key: Bin, iv: Bin = None, key_size=16, mode="ECB") -> Bin:
        self.check_key_size(key)
        plaintext_bin = self.convert_to_bin(plaintext)
        return self.aes_ecb_encrypt(plaintext_bin, key)

    def check_key_size(self, key: Bin):
        if key.size != self.key_size:
            raise ValueError(
                f"Key does not match the expected size for AES-{self.key_size}"
            )

In [54]:
# Cria instância do AES
aes = S_AES()

# Exemplo de chave e texto puro
example_key = Bin(0xA73B, 16)
example_plt = "ok"
expected_ciphertext = Bin(0x0738, 16)

# Realiza a criptografia
obtained_ciphertext = aes.encrypt(example_plt, example_key)

# Exibe os resultados em diferentes formatos
print("=== RESULTADOS DO TESTE ===")
print(f"Esperado (Bin):    {expected_ciphertext}")
print(f"Obtido   (Bin):    {obtained_ciphertext}")
print(f"Esperado (base64): {format(expected_ciphertext,'base64')}")
print(f"Obtido   (base64): {format(obtained_ciphertext,'base64')}")

# Realiza asserção com mensagem clara
assert obtained_ciphertext == expected_ciphertext, (
    f"Falha no teste: Ciphertext diferente!\n"
    f"Esperado: {expected_ciphertext}\nObtido: {obtained_ciphertext}"
)

# Exibe os logs (se houver)
print("\n=== LOGS GERADOS ===")
aes.logger.show()

=== RESULTADOS DO TESTE ===
Esperado (Bin):    0x0738
Obtido   (Bin):    0x0738
Esperado (base64): Bzg=
Obtido   (base64): Bzg=

=== LOGS GERADOS ===
=== Logs for phase 'Key Expansion' ===

[1] Round Key 0:
0xA 0x3
0x7 0xB

[2] Round Key 1:
0x1 0x2
0xC 0x7

[3] Round Key 2:
0x7 0x5
0x6 0x1

=== Logs for phase 'round 0' ===

[1] Plaintext block:
0x6 0x6
0xF 0xB

[2] Initial round key:
0xA 0x3
0x7 0xB

[3] After AddRoundKey:
0xC 0x5
0x8 0x0

=== Logs for phase 'round 1' ===

[1] Round key:
0x1 0x2
0xC 0x7

[2] Before Transformations:
0xC 0x5
0x8 0x0

[3] After SubBytes:
0xC 0x1
0x6 0x9

[4] After ShiftRows:
0xC 0x1
0x9 0x6

[5] After MixColumns:
0xE 0xA
0xC 0x2

[6] After AddRoundKey:
0xF 0x8
0x0 0x5

=== Logs for phase 'round 2' ===

[1] Round key:
0x7 0x5
0x6 0x1

[2] Before Transformations:
0xF 0x8
0x0 0x5

[3] After SubBytes:
0x7 0x6
0x9 0x1

[4] After ShiftRows:
0x7 0x6
0x1 0x9

[5] After AddRoundKey:
0x0 0x3
0x7 0x8



In [55]:
from Crypto.Cipher import AES
from Crypto.Util import Counter


class AESHelper:
    def __init__(self, key_size=16):
        self.key_size = key_size  # in bytes (16=128 bits)

    def check_key_size(self, key: bytes):
        if len(key) != self.key_size:
            raise ValueError(
                f"Key does not match the expected size for AES-{self.key_size * 8}")

    def aes_ecb_encrypt(self, plaintext: bytes, key: bytes) -> bytes:
        cipher = AES.new(key, AES.MODE_ECB)
        return cipher.encrypt(plaintext)

    def aes_cbc_encrypt(self, plaintext: bytes, key: bytes, iv: bytes) -> bytes:
        cipher = AES.new(key, AES.MODE_CBC, iv)
        return cipher.encrypt(plaintext)

    def aes_cfb_encrypt(self, plaintext: bytes, key: bytes, iv: bytes) -> bytes:
        cipher = AES.new(key, AES.MODE_CFB, iv)
        return cipher.encrypt(plaintext)

    def aes_ofb_encrypt(self, plaintext: bytes, key: bytes, iv: bytes) -> bytes:
        cipher = AES.new(key, AES.MODE_OFB, iv)
        return cipher.encrypt(plaintext)

    def aes_ctr_encrypt(self, plaintext: bytes, key: bytes, nonce: bytes) -> bytes:
        ctr = Counter.new(64, prefix=nonce, initial_value=0)
        cipher = AES.new(key, AES.MODE_CTR, counter=ctr)
        return cipher.encrypt(plaintext)

    def encrypt(self, plaintext: bytes, key: bytes, iv: bytes = None, mode="ECB", nonce: bytes = None) -> bytes:
        self.check_key_size(key)

        if mode == "ECB":
            return self.aes_ecb_encrypt(plaintext, key)
        elif mode == "CBC":
            if iv is None:
                raise ValueError("IV is required for CBC mode")
            return self.aes_cbc_encrypt(plaintext, key, iv)
        elif mode == "CFB":
            if iv is None:
                raise ValueError("IV is required for CFB mode")
            return self.aes_cfb_encrypt(plaintext, key, iv)
        elif mode == "OFB":
            if iv is None:
                raise ValueError("IV is required for OFB mode")
            return self.aes_ofb_encrypt(plaintext, key, iv)
        elif mode == "CTR":
            if nonce is None:
                raise ValueError("Nonce is required for CTR mode")
            return self.aes_ctr_encrypt(plaintext, key, nonce)

        raise ValueError(f"Unsupported AES mode: {mode}")

ModuleNotFoundError: No module named 'Crypto'

## Conclusão